In [1]:
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd

all_chars = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789!@#$%^&*?/\\ ")
char2idx = {c: i+1 for i, c in enumerate(all_chars)}

def encode_password(pw, max_len=40):
    pw = pw[:max_len]
    seq = [char2idx.get(c, 0) for c in pw]
    seq += [0] * (max_len - len(seq))
    return seq

class PasswordDataset(Dataset):
    def __init__(self, csv_path):
        df = pd.read_csv(csv_path)
        self.X = [encode_password(pw) for pw in df['password']]
        label_map = {"weak": 0, "normal": 1, "strong": 2}
        self.y = [label_map[label] for label in df['label']]
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])


In [2]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (hn, _) = self.lstm(x)
        out = self.fc(hn[-1])
        return out


In [3]:
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = PasswordDataset("balanced_passwords_re.csv")
train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64)

model = LSTMClassifier(vocab_size=len(char2idx) + 1).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_epoch():
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Train Loss: {total_loss / len(train_loader):.4f}")

def validate():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            preds = torch.argmax(out, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    print(f"Validation Accuracy: {correct / total:.4f}")

for epoch in range(1, 11):
    print(f"Epoch {epoch}")
    train_epoch()
    validate()


Epoch 1
Train Loss: 0.2939
Validation Accuracy: 0.9529
Epoch 2
Train Loss: 0.0889
Validation Accuracy: 0.9805
Epoch 3
Train Loss: 0.0574
Validation Accuracy: 0.9856
Epoch 4
Train Loss: 0.0366
Validation Accuracy: 0.9929
Epoch 5
Train Loss: 0.0238
Validation Accuracy: 0.9538
Epoch 6
Train Loss: 0.0156
Validation Accuracy: 0.9976
Epoch 7
Train Loss: 0.0094
Validation Accuracy: 0.9978
Epoch 8
Train Loss: 0.0065
Validation Accuracy: 0.9981
Epoch 9
Train Loss: 0.0056
Validation Accuracy: 0.9988
Epoch 10
Train Loss: 0.0039
Validation Accuracy: 0.9993


In [4]:
torch.save(model.state_dict(), "lstm_password_classifier_76941.pt")